# Data Cleaning & Set Up

In [38]:

import pandas as pd
import sqlite3


In [39]:
conn = sqlite3.connect('loveisblind.db')
%load_ext sql
%sql sqlite:///loveisblind.db



The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [46]:
#create table for contestants
df = pd.read_csv('contestants.csv')
contestants_table = df.to_sql('contestants', conn, if_exists='replace', index=False)
#create table for relationships
df = pd.read_csv('relationships.csv')
relationships_table = df.to_sql('relationships', conn, if_exists='replace', index=False)


In [41]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [42]:
%%sql
SELECT * FROM contestants LIMIT 2;


 * sqlite:///loveisblind.db
Done.


contestant_id,name,season,age,gender,occupation,city,has_children 0=no
C001,Alyson Costa,1,36,F,Pediatric Nurse Client Relations Director,Atlanta,0
C002,Amber Pike,1,27,F,Ex-Tank Mechanic,Atlanta,0


## Data Cleaning for Contestant Data

In [43]:
%%sql
--I'm trimming and normalizeing the names, gender, city, 
--and occupation to lowercase. 
UPDATE contestants 
SET name = TRIM(LOWER(name)),
 gender = TRIM(lower(gender)),
 city = TRIM(lower(city)),
 occupation = TRIM(lower(occupation));
--show results
SELECT * from contestants LIMIT 5;


 * sqlite:///loveisblind.db
300 rows affected.
Done.


contestant_id,name,season,age,gender,occupation,city,has_children 0=no
C001,alyson costa,1,36,f,pediatric nurse client relations director,atlanta,0
C002,amber pike,1,27,f,ex-tank mechanic,atlanta,0
C003,andy rickert,1,30,m,welder,atlanta,0
C004,briana holmes,1,24,f,ceo,atlanta,0
C005,cameron hamilton,1,28,m,data scientist,atlanta,0


In [47]:
%%sql
SELECT * FROM relationships LIMIT 2;

 * sqlite:///loveisblind.db
Done.


relationship_id,female_id,male_id,made_it_to_the_alter,sex_during_honeymoon,season,age_difference,married,love_triange,engaged,together_after_1_year
1,C002,C024,Y,Y,1,0,Y,Y,Y,Y
2,C020,C005,Y,Y,1,4,Y,N,Y,Y


## Analyze Data Using SQL

### Question 1: Who is the youngest Contestant?

In [ ]:
%%sql
SELECT  name, season ,MIN(age) AS youngest_contestant_age
FROM contestants;


 * sqlite:///loveisblind.db
Done.


name,season,youngest_contestant_age
briana holmes,1,24


In [88]:
%%sql
SELECT  
        season, 
        MIN(age) AS youngest_contestant_age
FROM contestants
GROUP BY season
ORDER BY youngest_contestant_age ASC;


 * sqlite:///loveisblind.db
Done.


season,youngest_contestant_age
1,24
5,24
6,24
3,25
4,26
7,26
8,26
2,27
9,27
10,28


### Question 2: Who is the oldest contestant?

In [ ]:
%%sql
SELECT  name,season, MAX(age) AS oldest_contestant_age
FROM contestants;

 * sqlite:///loveisblind.db
Done.


name,season,oldest_contestant_age
amanda burke,8,43


In [89]:
%%sql
SELECT  
        season, 
        MAX(age) AS oldest_contestant_age
FROM contestants
GROUP BY season
ORDER BY oldest_contestant_age ASC;

 * sqlite:///loveisblind.db
Done.


season,oldest_contestant_age
5,34
3,36
6,36
7,37
1,38
2,39
4,39
10,39
9,41
8,43


### Question 3: What is the average age of ALL contestants?

In [ ]:
%%sql

SELECT AVG(age) as average_age_of_all_contestants
FROM contestants;

 * sqlite:///loveisblind.db
Done.


average_age_of_all_contestants
31.116666666666667


### Question 4: What is the average age of contestants by season?

In [74]:
%%sql
--
SELECT season, round(AVG(age), 0) as average_age
FROM contestants
GROUP BY season
ORDER BY average_age DESC;


 * sqlite:///loveisblind.db
Done.


season,average_age
10,33.0
9,32.0
7,32.0
8,31.0
4,31.0
3,31.0
2,31.0
6,30.0
5,30.0
1,30.0


What we can gather from this data is that for the most part average age increased as the show went on. Earlier we saw the youngest age did increase as the seasons went on. Oldest Contestant increased as the seasons went by. 

### Question 5: How many contestants are in each season?

In [ ]:
%%sql 
SELECT SEASON, COUNT(*) AS num_contestants
FROM contestants
GROUP BY season;

 * sqlite:///loveisblind.db
Done.


season,num_contestants
1,29
2,29
3,30
4,30
5,28
6,29
7,29
8,32
9,32
10,32


## Taking a look at Relationships Table
I wanted to do this join to see all the couples from the show and the final outcome of their relationship. This is to help with creating and understanding questions I'm going to ask about the relationship Table

In [23]:
%%sql 

SELECT 
    r.relationship_id, female.name as feamle_name, 
    male.name as male_name,engaged,  married

FROM relationships r
JOIN contestants female 
ON r.female_id = female.contestant_id
JOIN contestants male
ON r.male_id = male.contestant_id;

 * sqlite:///loveisblind.db
Done.


relationship_id,feamle_name,male_name,engaged,married
1,Amber Pike,Matthew Barnett,Y,Y
2,Lauren Speed,Cameron Hamilton,Y,Y
3,Diamond Jack,Carlton Morton,Y,N
4,Giannina Gibelli,Damian Powers,Y,N
5,Danielle Drouin,Rory Newbrough,Y,N
6,Carlton Morton,Mark Cuevas,Y,N
7,Kelly Chase,Kenny Barnes,Y,N
8,Lexie Skipper,Westley Baer,Y,N
9,Deepti Vempati,Abhishek Chatterjee,Y,N
10,Caitlin McKee,James Miller,Y,N


### Question 7: What percentage of contestants got engaged?

In [ ]:
%%sql
SELECT COUNT(*) AS 
total_contestants
FROM contestants;



 * sqlite:///loveisblind.db
Done.


total_contestants
300


In [20]:
%%sql
SELECT COUNT(*) AS 
engaged_contestants
FROM (
    SELECT female_id as contestant_id
    FROM relationships

    UNION
    SELECT male_id as contestant_id
    FROM relationships
) AS gotEngaged;



 * sqlite:///loveisblind.db
Done.


engaged_contestants
124


In [28]:
#percentage of contestants that got engaged: 
percentageOfEngagement = (124/300) * 100
print(round(percentageOfEngagement,2))

41.33


### Question 8: What Season has the most Engagements

In [57]:
%%sql
SELECT season, COUNT(*) as Num_of_Engagements
FROM relationships
GROUP BY season
ORDER BY num_of_engagements DESC

 * sqlite:///loveisblind.db
Done.


season,Num_of_Engagements
4,9
2,8
1,8
10,7
7,7
9,6
8,5
6,5
3,5
5,4


Based on these results there was something about Season 4 that had engagements up. Engagments were very low for season 5. It would be smart to look at different things about these 2 seasons to see what might cause a huge increase in enegagments. 

### Question 9: What Percentage of engagements resulted in marriage

In [ ]:
%%sql
--SELECT COUNT(*) as numEngagements
--FROM relationships
--SELECT COUNT(*) as numMarried
--FROM relationships
--WHERE married = 'Y';

--Realized after more learning that this can be put into one query.
SELECT
    COUNT(*) as numEngagments, 
    SUM(CASE WHEN married = 'Y' THEN 1 ELSE 0 END) AS numMarried, 
    SUM(CASE WHEN married = 'Y' THEN 1 ELSE 0 END)* 100.0 / COUNT(*) as percentageMarried

FROM relationships


 * sqlite:///loveisblind.db
Done.


numEngagments,numMarried,percentageMarried
64,16,25.0


I want to see this now by season. The percentage of marriages is pretty low. Only 25% got married.

### Question 10: What is the percentage of engaged couples that got Married for each season?

In [72]:
%%sql
SELECT
    season,
    COUNT(*) as numEngagments, 
    SUM(CASE WHEN married = 'Y' THEN 1 ELSE 0 END) AS numMarried, 
    round(SUM(CASE WHEN married = 'Y' THEN 1 ELSE 0 END)* 100.0 / COUNT(*), 2)as percentageMarried

FROM relationships
GROUP BY season
ORDER BY percentageMarried DESC

 * sqlite:///loveisblind.db
Done.


season,numEngagments,numMarried,percentageMarried
3,5,2,40.0
4,9,3,33.33
10,7,2,28.57
7,7,2,28.57
5,4,1,25.0
2,8,2,25.0
1,8,2,25.0
8,5,1,20.0
6,5,1,20.0
9,6,0,0.0


Season 3 was one of the lowest number of Engagements but had the highest percentage of marriages out off the seasons. Where season 4 had a high number of engagements and a high percentage of people married. 

Season 9 was a historic season where none of the engaged couples got married.

In my next section of Analysis I want to take a look at Season 9, 3, 4, and 5